# DiffProtect Colab Runner

這份 notebook 會在 Google Colab GPU 上執行 `joellliu/DiffProtect`。

流程：
1. 開啟 Colab GPU runtime
2. 安裝與修補 DiffProtect 需要的套件
3. 下載 DiffAE checkpoint 與 victim face model 權重
4. 上傳自拍照
5. 對齊臉部並產生保護後圖片
6. 打包下載結果

> 注意：DiffProtect 沒有官方 Colab，這份 notebook 會盡量自動化，但 Google Drive 權重連結若被限流，可能需要手動下載後上傳。

## 0. Colab 設定

先到 Colab 選單：`Runtime` -> `Change runtime type` -> `T4 GPU` 或其他 GPU。

In [ ]:
import os, sys, subprocess, textwrap, json, shutil, glob
from pathlib import Path

def run(cmd, check=True):
    print(f'$ {cmd}')
    return subprocess.run(cmd, shell=True, check=check)

try:
    import torch
    print('Python:', sys.version)
    print('Torch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
    else:
        raise RuntimeError('請先把 Colab runtime 切到 GPU。')
except Exception as exc:
    raise RuntimeError('GPU 檢查失敗，請確認 Colab runtime 已啟用 GPU。') from exc

## 1. 下載 DiffProtect 原始碼

In [ ]:
%cd /content
!rm -rf DiffProtect
!git clone https://github.com/joellliu/DiffProtect.git
%cd /content/DiffProtect
!git rev-parse --short HEAD

## 2. 安裝套件

DiffProtect 的原始 `requirements.txt` 釘在很舊的 Python / PyTorch 版本。Colab 目前通常是 Python 3.10/3.11 + 新版 PyTorch，所以這裡保留 Colab 內建 PyTorch，並安裝其餘依賴與相容性修補。

In [ ]:
!python -m pip install -q --upgrade pip
!python -m pip uninstall -y -q torchtext || true
!python -m pip install -q --force-reinstall --no-cache-dir 'numpy==1.26.4' 'scipy==1.11.4' 'pandas==2.2.2'
!python -m pip install -q --force-reinstall --no-cache-dir 'opencv-python==4.10.0.84' 'opencv-python-headless==4.10.0.84'
!python -m pip install -q pillow tqdm gdown requests dlib
!python -m pip install -q --force-reinstall --no-cache-dir advertorch
!python -m pip install -q pytorch-lightning==1.4.5 torchmetrics==0.5.0 pytorch-fid==0.2.0 lpips==0.1.4 lmdb==1.2.1 ftfy regex
!python -m pip install -q --force-reinstall --no-cache-dir 'numpy==1.26.4' 'scipy==1.11.4' 'pandas==2.2.2'
!python -m pip install -q --no-deps facenet-pytorch

# numpy / PyTorch 新版相容性修補，讓舊 checkpoint 和舊套件比較容易載入。
Path('sitecustomize.py').write_text('''
import numpy as _np
for _name, _value in {\"complex\": complex, \"float\": float, \"int\": int, \"bool\": bool}.items():
    if not hasattr(_np, _name):
        setattr(_np, _name, _value)

try:
    import torch as _torch
    import torch.autograd.gradcheck as _gradcheck
    if not hasattr(_gradcheck, \"zero_gradients\"):
        def zero_gradients(inputs):
            if isinstance(inputs, _torch.Tensor):
                if inputs.grad is not None:
                    inputs.grad.detach_()
                    inputs.grad.zero_()
            else:
                for elem in inputs:
                    zero_gradients(elem)
        _gradcheck.zero_gradients = zero_gradients
except Exception:
    pass
'''.strip() + '\
')

# advertorch 0.2 會匯入新版 PyTorch 已移除的 zero_gradients；直接 patch site-packages，讓 subprocess 也生效。
import sysconfig
site_packages = Path(sysconfig.get_paths()['purelib'])
advertorch_dir = site_packages / 'advertorch'
zero_gradients_patch = """try:
    from torch.autograd.gradcheck import zero_gradients
except ImportError:
    import torch
    def zero_gradients(inputs):
        if isinstance(inputs, torch.Tensor):
            if inputs.grad is not None:
                inputs.grad.detach_()
                inputs.grad.zero_()
        else:
            for elem in inputs:
                zero_gradients(elem)"""
if advertorch_dir.exists():
    for patch_file in advertorch_dir.rglob('*.py'):
        text = patch_file.read_text()
        old = 'from torch.autograd.gradcheck import zero_gradients'
        if old in text:
            lines = text.splitlines()
            fixed_lines = []
            for line in lines:
                if old in line:
                    indent = line[:len(line) - len(line.lstrip())]
                    fixed_lines.extend(indent + patch_line if patch_line else patch_line for patch_line in zero_gradients_patch.splitlines())
                else:
                    fixed_lines.append(line)
            patch_file.write_text('\n'.join(fixed_lines) + '\n')
            print('Patched advertorch:', patch_file)

# 不在目前 notebook kernel import advertorch；pip 更換 numpy 後，舊 kernel 可能還保留 ABI 不相容狀態。
# demo.py 會用新的 Python subprocess 載入修補後的套件。
validation = subprocess.run(
    "python - <<'PY'\nfrom advertorch.utils import clamp\nimport numpy as np\nprint('advertorch subprocess import ok; numpy', np.__version__)\nPY",
    shell=True,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
print(validation.stdout)
if validation.returncode != 0:
    raise RuntimeError('advertorch subprocess import validation failed')

# torch 2.6+ 的 torch.load 預設 weights_only=True，舊 Lightning checkpoint 需要關掉。
demo_path = Path('demo.py')
demo_text = demo_path.read_text()
demo_text = demo_text.replace("torch.load(f'checkpoints/{conf.name}/last.ckpt', map_location='cpu')", "torch.load(f'checkpoints/{conf.name}/last.ckpt', map_location='cpu', weights_only=False)")
demo_path.write_text(demo_text)

# DiffProtect 的 assets/ 被官方 repo gitignore，Colab clone 後常缺 assets.models.* 原始碼。
# 這裡把 mobile_face 快速模式改成 facenet-pytorch 的預訓練 InceptionResnetV1 surrogate，避免依賴缺失的 assets.models package。
attack_utils_path = Path('attack_utils.py')
attack_utils_text = attack_utils_path.read_text()
attack_utils_text = attack_utils_text.replace(
    'from assets.models import irse, ir152, facenet\n',
    'try:\n    from assets.models import irse, ir152, facenet\nexcept Exception:\n    irse = ir152 = facenet = None\nfrom facenet_pytorch import InceptionResnetV1\n'
)
old_mobile_face = """if model_name == 'mobile_face':
            test_models[model_name] = []
            test_models[model_name].append((112, 112))
            fr_model = irse.MobileFaceNet(512)
            fr_model.load_state_dict(torch.load('./assets/models/mobile_face.pth'))
            fr_model.to(device)
            fr_model.eval()
            test_models[model_name].append(fr_model)"""
new_mobile_face = """if model_name == 'mobile_face':
            test_models[model_name] = []
            test_models[model_name].append((160, 160))
            fr_model = InceptionResnetV1(pretrained='vggface2', classify=False, device=device)
            fr_model.to(device)
            fr_model.eval()
            test_models[model_name].append(fr_model)"""
if old_mobile_face in attack_utils_text:
    attack_utils_text = attack_utils_text.replace(old_mobile_face, new_mobile_face)
else:
    print('Warning: mobile_face block not found; attack_utils.py may have changed upstream.')
attack_utils_path.write_text(attack_utils_text)

validation = subprocess.run(
    "python - <<'PY'\nimport attack_utils\nprint('attack_utils import ok')\nPY",
    shell=True,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
print(validation.stdout)
if validation.returncode != 0:
    raise RuntimeError('attack_utils import validation failed')

print('Dependencies installed and compatibility patches applied.')

## 3. 下載模型權重

需要兩組權重：
- DiffAE `ffhq256_autoenc/last.ckpt`
- DiffProtect victim face models

如果 Google Drive 被限流，這格可能失敗。失敗時請照下一格的手動上傳方式。

In [ ]:
from pathlib import Path
import shutil
Path('checkpoints/ffhq256_autoenc').mkdir(parents=True, exist_ok=True)
Path('assets/models').mkdir(parents=True, exist_ok=True)

def normalize_victim_model_paths():
    expected = ['mobile_face.pth', 'ir152.pth', 'irse50.pth', 'facenet.pth']
    models_dir = Path('assets/models')
    models_dir.mkdir(parents=True, exist_ok=True)
    search_roots = [Path('assets'), Path('/content')]

    for filename in expected:
        dst = models_dir / filename
        if dst.exists():
            continue
        matches = []
        for root in search_roots:
            if root.exists():
                matches.extend([p for p in root.rglob(filename) if p.is_file()])
        if matches:
            src = matches[0]
            if src.resolve() != dst.resolve():
                shutil.copy2(src, dst)
                print(f'Copied {src} -> {dst}')

    return {name: (models_dir / name).exists() for name in expected}

# DiffAE FFHQ256 autoencoder checkpoint folder
if not Path('checkpoints/ffhq256_autoenc/last.ckpt').exists():
    run("gdown --folder 'https://drive.google.com/drive/folders/1-5zfxT6Gl-GjxM7z9ZO2AHlB70tfmF6V' -O checkpoints/ffhq256_autoenc --remaining-ok", check=False)

# Victim model weights from DiffProtect README
if not Path('assets/models/mobile_face.pth').exists():
    run('rm -rf /content/diffprotect_assets_extract')
    run('mkdir -p /content/diffprotect_assets_extract')
    run("gdown --fuzzy 'https://drive.google.com/file/d/19_Y0jR789BGciogjjoGtWNEv-5QBiCB7/view?usp=sharing' -O /content/diffprotect_assets.zip", check=False)
    if Path('/content/diffprotect_assets.zip').exists():
        run('unzip -q -o /content/diffprotect_assets.zip -d /content/diffprotect_assets_extract', check=False)
        run('cp -R /content/diffprotect_assets_extract/* assets/ 2>/dev/null || true', check=False)
    else:
        print('victim model zip 沒有下載成功，可能是 Google Drive 限流或權限問題。')

model_status = normalize_victim_model_paths()
print('Victim model status:', model_status)

print('DiffAE checkpoint exists:', Path('checkpoints/ffhq256_autoenc/last.ckpt').exists())
print('MobileFaceNet weight exists:', Path('assets/models/mobile_face.pth').exists())

if not Path('assets/models/mobile_face.pth').exists():
    print('找不到 mobile_face.pth。請先看上方 gdown/unzip 是否失敗；若 Google Drive 限流，請改跑下一格手動上傳 victim models zip。')

### 如果自動下載失敗，手動上傳權重

只有在上一格失敗時才需要跑這格。

- 把 DiffAE 的 `last.ckpt` 上傳，會放到 `checkpoints/ffhq256_autoenc/last.ckpt`
- 把 victim model zip 上傳，會解壓到 `assets/`

In [ ]:
RUN_MANUAL_WEIGHT_UPLOAD = False  # 自動下載失敗時改成 True 再跑這格

if RUN_MANUAL_WEIGHT_UPLOAD:
    from google.colab import files
    print('請上傳 DiffAE last.ckpt')
    uploaded = files.upload()
    for name in uploaded:
        if name.endswith('.ckpt'):
            Path('checkpoints/ffhq256_autoenc').mkdir(parents=True, exist_ok=True)
            shutil.move(name, 'checkpoints/ffhq256_autoenc/last.ckpt')

    print('請上傳 victim models zip')
    uploaded = files.upload()
    for name in uploaded:
        if name.endswith('.zip'):
            run(f'rm -rf /content/diffprotect_manual_assets_extract && mkdir -p /content/diffprotect_manual_assets_extract')
            run(f'unzip -q -o {name} -d /content/diffprotect_manual_assets_extract')
            run('cp -R /content/diffprotect_manual_assets_extract/* assets/ 2>/dev/null || true')
            if 'normalize_victim_model_paths' in globals():
                print('Victim model status:', normalize_victim_model_paths())

print('DiffAE checkpoint exists:', Path('checkpoints/ffhq256_autoenc/last.ckpt').exists())
print('MobileFaceNet weight exists:', Path('assets/models/mobile_face.pth').exists())

## 4. 上傳自拍照

可以一次上傳多張。建議使用正臉、清楚、單人照片。

In [ ]:
from google.colab import files
from pathlib import Path
import shutil

raw_dir = Path('/content/diffprotect_input_raw')
aligned_dir = Path('/content/diffprotect_input_aligned')
raw_dir.mkdir(parents=True, exist_ok=True)
aligned_dir.mkdir(parents=True, exist_ok=True)

uploaded = files.upload()
for name in uploaded:
    suffix = Path(name).suffix.lower()
    if suffix in ['.jpg', '.jpeg', '.png', '.webp']:
        shutil.move(name, raw_dir / name)

print('Uploaded images:')
for p in sorted(raw_dir.glob('*')):
    print('-', p.name)

## 5. 對齊臉部到 256x256

DiffProtect / DiffAE 預期輸入是 FFHQ 風格的 256x256 對齊臉。

這格會先使用 DiffProtect 原本的 dlib/FFHQ 對齊；如果偵測不到臉，會自動改用 OpenCV 臉部裁切，最後再用中心裁切保底，避免流程卡住。

In [ ]:
!python align.py -i /content/diffprotect_input_raw -o /content/diffprotect_input_aligned || true

from PIL import Image, ImageOps
import cv2
import numpy as np

def fallback_align_image(src_path, dst_path):
    img = Image.open(src_path).convert('RGB')
    img = ImageOps.exif_transpose(img)
    w, h = img.size

    arr = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
    gray = cv2.cvtColor(arr, cv2.COLOR_BGR2GRAY)
    detector = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    faces = detector.detectMultiScale(gray, scaleFactor=1.08, minNeighbors=3, minSize=(40, 40))

    if len(faces):
        x, y, fw, fh = max(faces, key=lambda f: f[2] * f[3])
        side = int(max(fw, fh) * 2.4)
        cx = x + fw / 2
        cy = y + fh * 0.55
        print(f'Fallback face crop: {src_path.name}, face=({x}, {y}, {fw}, {fh})')
    else:
        side = min(w, h)
        cx = w / 2
        cy = h / 2
        print(f'Fallback center crop: {src_path.name} (OpenCV 也沒有偵測到臉)')

    side = max(64, min(side, max(w, h)))
    left = int(round(cx - side / 2))
    top = int(round(cy - side / 2))
    right = left + side
    bottom = top + side

    pad_left = max(0, -left)
    pad_top = max(0, -top)
    pad_right = max(0, right - w)
    pad_bottom = max(0, bottom - h)
    if any([pad_left, pad_top, pad_right, pad_bottom]):
        img = ImageOps.expand(img, (pad_left, pad_top, pad_right, pad_bottom), fill=(0, 0, 0))
        left += pad_left
        top += pad_top
        right += pad_left
        bottom += pad_top

    crop = img.crop((left, top, right, bottom)).resize((256, 256), Image.Resampling.LANCZOS)
    crop.save(dst_path)

aligned = sorted(Path('/content/diffprotect_input_aligned').glob('*'))
if not aligned:
    print('dlib/FFHQ 對齊沒有輸出，改用 fallback crop 產生 256x256 輸入。')
    aligned_dir.mkdir(parents=True, exist_ok=True)
    for src in sorted(raw_dir.glob('*')):
        if src.suffix.lower() in ['.jpg', '.jpeg', '.png', '.webp']:
            dst = aligned_dir / f'{src.stem}.png'
            fallback_align_image(src, dst)
    aligned = sorted(Path('/content/diffprotect_input_aligned').glob('*'))

print('Aligned images:', len(aligned))
for p in aligned:
    print('-', p.name)

if not aligned:
    raise RuntimeError('沒有產生任何 256x256 輸入圖。請確認上傳的是 jpg/png/webp 照片。')

## 6. 準備 target face

DiffProtect 是 targeted attack，需要一張 target face。這份 notebook 會優先使用 DiffProtect asset 裡的預設 target；如果不存在，會下載 NVIDIA FFHQ repo 的公開範例圖，裁一張臉作為 target。

> 若你想要更完整的保護效果，可以把 `USE_CUSTOM_TARGET` 改成 `True`，另外上傳一張 target 臉。

In [ ]:
USE_CUSTOM_TARGET = False

target_path = Path('assets/datasets/target/085807.jpg')
public_target_raw = Path('/content/diffprotect_public_target_raw.png')
public_target_aligned = Path('/content/diffprotect_public_target.png')

if USE_CUSTOM_TARGET:
    from google.colab import files
    custom_raw = Path('/content/diffprotect_target_raw')
    custom_aligned = Path('/content/diffprotect_target_aligned')
    custom_raw.mkdir(parents=True, exist_ok=True)
    custom_aligned.mkdir(parents=True, exist_ok=True)
    print('請上傳一張 target face')
    uploaded = files.upload()
    for name in uploaded:
        suffix = Path(name).suffix.lower()
        if suffix in ['.jpg', '.jpeg', '.png', '.webp']:
            shutil.move(name, custom_raw / name)
    run('python align.py -i /content/diffprotect_target_raw -o /content/diffprotect_target_aligned', check=False)
    targets = sorted(custom_aligned.glob('*'))
    if not targets:
        print('dlib/FFHQ 對齊 target 沒有輸出，改用 fallback crop 產生 256x256 target。')
        for src in sorted(custom_raw.glob('*')):
            if src.suffix.lower() in ['.jpg', '.jpeg', '.png', '.webp']:
                dst = custom_aligned / f'{src.stem}.png'
                fallback_align_image(src, dst)
        targets = sorted(custom_aligned.glob('*'))
    if not targets:
        raise RuntimeError('target face 對齊失敗，請換一張清楚正臉照片。')
    target_path = targets[0]
elif not target_path.exists():
    print('找不到 asset 預設 target，改用公開 FFHQ 範例圖裁切 target face。')
    if not public_target_raw.exists():
        run('curl -L https://raw.githubusercontent.com/NVlabs/ffhq-dataset/master/ffhq-teaser.png -o /content/diffprotect_public_target_raw.png')
    fallback_align_image(public_target_raw, public_target_aligned)
    target_path = public_target_aligned

if Path(target_path).resolve() in [p.resolve() for p in sorted(Path('/content/diffprotect_input_aligned').glob('*'))]:
    raise RuntimeError('target_path 不應該和 source 自拍照相同。請設定 USE_CUSTOM_TARGET=True 並上傳 target，或確認公開 target 下載成功。')

print('Target path:', target_path)

## 7. 執行 DiffProtect

預設使用 `mobile_face` 名稱，但 Colab 版會把它 patch 成 `facenet-pytorch` 的預訓練 InceptionResnetV1 surrogate，避免官方 repo 缺少 `assets.models` 原始碼時無法執行。

In [ ]:
ATTACK_MODEL_NAMES = ['mobile_face']
TEST_MODEL_NAMES = ['mobile_face']

# 快速設定。想要更接近論文預設，可把 ITER=50, T_ENC=100, T_INF=100，但會更慢。
ITER = 20
T_ENC = 50
T_ATK = 5
T_INF = 50

save_root = '/content/diffprotect_outputs'
demo_log = Path('/content/diffprotect_demo.log')

required_paths = [
    Path('checkpoints/ffhq256_autoenc/last.ckpt'),
    Path(target_path),
]
if not Path('assets/models/mobile_face.pth').exists() and 'normalize_victim_model_paths' in globals():
    print('mobile_face.pth 不在預期位置，嘗試整理已下載/已解壓的權重檔...')
    print('Victim model status:', normalize_victim_model_paths())

missing = [str(p) for p in required_paths if not p.exists()]
if missing:
    raise RuntimeError('缺少必要檔案，請回到「3. 下載模型權重」cell 重跑；若 Google Drive 限流，改跑手動上傳權重 cell：\n' + '\n'.join(missing))

source_images = sorted(Path('/content/diffprotect_input_aligned').glob('*'))
if not source_images:
    raise RuntimeError('沒有 aligned source images，請回到「對齊臉部」cell 重跑。')

print('Source images:', [p.name for p in source_images])
print('Target:', target_path)

cmd = [
    'python demo.py',
    '--source_dir /content/diffprotect_input_aligned',
    f'--save_path {save_root}',
    '--clean_path /content/diffprotect_input_aligned',
    f'--target_path {target_path}',
    f'--test_path {target_path}',
    '--model_names ' + ' '.join(ATTACK_MODEL_NAMES),
    '--test_model_names ' + ' '.join(TEST_MODEL_NAMES),
    f'--iter {ITER}',
    f'--T_enc {T_ENC}',
    f'--T_atk {T_ATK}',
    f'--T_inf {T_INF}',
]

cmd_str = ' '.join(cmd)
print(f'$ {cmd_str}')
proc = subprocess.run(cmd_str, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
demo_log.write_text(proc.stdout)

lines = proc.stdout.splitlines()
print('\n'.join(lines[-120:]))
print(f'\nFull log saved to: {demo_log}')

if proc.returncode != 0:
    raise RuntimeError(f'DiffProtect 執行失敗，exit code={proc.returncode}。請把上面最後 120 行 log 貼給我。')

## 8. 預覽與下載結果

In [ ]:
from IPython.display import display
from PIL import Image
from google.colab import files

output_dirs = sorted(Path('/content/diffprotect_outputs').glob('save_*'))
if not output_dirs:
    raise RuntimeError('找不到輸出資料夾，請確認上一格是否成功。')

latest_output = output_dirs[-1]
print('Latest output:', latest_output)

pngs = sorted(latest_output.glob('*.png'))
print('Protected images:', len(pngs))
for p in pngs[:6]:
    print(p.name)
    display(Image.open(p).resize((256, 256)))

zip_path = '/content/diffprotect_results.zip'
!rm -f /content/diffprotect_results.zip
!zip -qr /content/diffprotect_results.zip {latest_output}
files.download(zip_path)

## 常見問題

- `CUDA available: False`：請切換 Colab GPU runtime 後從頭重跑。
- Google Drive 權重下載失敗：通常是配額或權限問題，改用手動上傳權重那格。
- 臉部對齊失敗：換正臉、清楚、單人照片。
- Colab 記憶體不足：維持 `mobile_face`，降低 `ITER/T_ENC/T_INF`，或改用 Colab Pro GPU。